# Recs 003: Query embed + top‑K retrieval

Uses **`recs_002`** artifacts: dense **L2-normalized** game vectors and `game_profile_embedding_meta.json` (same **TF Hub** URL as indexing).

**Flow:** embed query text → L2-normalize → **dot product** with the game matrix (= cosine similarity) → **top‑K** `app_id` / `app_name`.

**Requires:** run [`recs_002_embed_game_profiles.ipynb`](./recs_002_embed_game_profiles.ipynb) first. Same conda env / `tensorflow-hub` as `recs_002`.

**Product path (later):** query string comes from `extract_preferences` + `build_embedding_input` ([`docs/recommender_transition_plan.md`](../../../docs/recommender_transition_plan.md)); this notebook uses **hand-written strings** for smoke tests and demos.

In [15]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")


REPO_ROOT = _repo_root()
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
NPZ_PATH = ARTIFACT_DIR / "game_profile_embeddings.npz"
INDEX_PATH = ARTIFACT_DIR / "game_profile_embedding_index.parquet"
META_PATH = ARTIFACT_DIR / "game_profile_embedding_meta.json"

for p in (NPZ_PATH, INDEX_PATH, META_PATH):
    if not p.is_file():
        raise FileNotFoundError(f"Run recs_002 first. Missing {p}")

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
TFHUB_URL = meta["model_name"]
EMBED_DIM = int(meta["dim"])
MAX_CHARS = meta.get("max_chars_per_review")

print("TF Hub:", TFHUB_URL)
print("dim:", EMBED_DIM, "n_games:", meta.get("n_games"))

TF Hub: https://tfhub.dev/google/universal-sentence-encoder/4
dim: 512 n_games: 315


In [16]:
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
import tensorflow_hub as hub

for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

embed_fn = hub.load(TFHUB_URL)

**Next:** run **load-index** (next cell) to load **`X`** and **`idx_df`**. The **Game catalog** section immediately after that lists every `app_id` / `app_name`.

*(Index file = Parquet, not JSON. Model metadata = read **`META_PATH`** with `json.load`.)*

In [17]:
z = np.load(NPZ_PATH)
X = np.asarray(z["embeddings"], dtype=np.float32)
app_ids_npz = np.asarray(z["app_id"], dtype=np.int64)
z.close()

idx_df = pd.read_parquet(INDEX_PATH)
if len(idx_df) != X.shape[0]:
    raise ValueError("Index rows must match embedding rows")
if not np.array_equal(idx_df["app_id"].to_numpy(), app_ids_npz):
    raise ValueError("app_id column must match npz app_id order")
if X.shape[1] != EMBED_DIM:
    raise ValueError("meta dim does not match embedding matrix")

print("X:", X.shape, "dtype:", X.dtype)

X: (315, 512) dtype: float32


### Game catalog (`app_id`, `app_name`)

**`idx_df`** — one row per game, same order as **`X`**. Sorted by **`app_name`** for browsing. *(Parquet source: `INDEX_PATH`.)*

In [29]:
from IPython.display import HTML, display

games_catalog = idx_df[["app_id", "app_name"]].sort_values("app_name", ignore_index=True)
print(f"{len(games_catalog)} games in index")

chunk_size = 10

# Add explanation for what the number is under the game name
explanation_html = (
    "<div style='margin-bottom:8px;'>"
    "<b>Note:</b> The number shown directly under each game name is the game's <b>Steam app_id</b>—a unique identifier used internally on Steam and in this recommender."
    "</div>"
)

def make_row(chunk):
    return "<tr>" + "".join(
        f"<td>{row.app_name}<br><span style='color:gray; font-size:small;'>{row.app_id}</span></td>"
        for _, row in chunk.iterrows()
    ) + "".join("<td></td>" for _ in range(chunk_size - len(chunk))) + "</tr>"

rows_html = "\n".join(
    make_row(games_catalog.iloc[i:i+chunk_size])
    for i in range(0, len(games_catalog), chunk_size)
)

header = "".join([f"<th>Game {i+1}</th>" for i in range(chunk_size)])
html_table = f"""
{explanation_html}
<table border="1" style="border-collapse:collapse">
<thead><tr>{header}</tr></thead>
<tbody>
{rows_html}
</tbody>
</table>
"""
display(HTML(html_table))


315 games in index


Game 1,Game 2,Game 3,Game 4,Game 5,Game 6,Game 7,Game 8,Game 9,Game 10
20XX322110,A Hat in Time253230,A Short Hike1055540,A Way Out1222700,ARK: Survival Evolved346110,ATLAS834910,Age of Empires II (2013)221380,Age of Empires: Definitive Edition1017900,American Truck Simulator270880,Among Us945360
Ancestors Legacy620590,Arma 3107410,Artifact583950,Assassin's Creed Odyssey812140,Assassin's Creed Origins582160,Avorion445220,Axiom Verge332200,BATTALION 1944489940,BATTLETECH637090,BERSERK and the Band of the Hawk502280
Baba Is You736260,Banished242920,Batman: Arkham Asylum GOTY Edition35140,Battle Royale Trainer772540,BattleBlock Theater238460,BeamNG.drive284160,Beat Saber620980,BioShock Infinite8870,Black Desert Online582660,Black Mesa362890
Blackwake420290,Bless Online681660,Bloons TD 6960090,Bomber Crew537800,Borderlands 3397540,Broforce274190,Budget Cuts400940,CHRONO TRIGGER613830,Call of Duty: Infinite Warfare292730,Call of Duty: WWII476600
Castle Crashers204360,Cave Story+200900,Celeste504230,Cities: Skylines255710,Clicker Heroes 2629910,Cold Waters541210,Conan Exiles440900,Counter-Strike: Source240,Crash Bandicoot™ N. Sane Trilogy731490,Crusader Kings III1158310
Cube World1128000,Cuphead268910,Cyberdimension Neptunia: 4 Goddesses Online632350,DARK SOULS™ III374320,DARK SOULS™: REMASTERED570940,DEATH STRANDING1190460,DOOM379720,DOOM Eternal782330,DRAGON BALL FighterZ678950,DRAGON QUEST HEROES™ II574050
DUSK519860,DYNASTY WARRIORS 9730310,Danganronpa 2: Goodbye Despair413420,Danganronpa: Trigger Happy Havoc413410,Darkest Dungeon®262060,Darksiders III606280,Day of Infamy447820,Dead Cells588650,Dead Rising 4543460,Dead by Daylight381210
Deep Rock Galactic548430,Desolate671510,Detention555220,Deus Ex: The Fall258180,Devil May Cry HD Collection631510,DiRT 4421020,Dishonored205100,Divinity: Original Sin 2435150,Doki Doki Literature Club698780,Don't Escape: 4 Days to Survive611760
Don't Starve219740,Don't Starve Together322330,Down To One334040,Dragon Cliff 龙崖758190,Duck Game312530,Due Process753650,Dungreed753420,Dying Light239140,Eco382310,Enter the Gungeon311690
Euro Truck Simulator 2227300,Europa Universalis IV236850,FAR: Lone Sails609320,FINAL FANTASY XII THE ZODIAC AGE595520,FINAL FANTASY XIV Online39210,FINAL FANTASY XV WINDOWS EDITION637650,FTL: Faster Than Light212680,Factorio427520,Fairy Fencer F Advent Dark Force524580,Fallout 4377160


In [19]:
# Explore the contents of the loaded npz file (`z` was loaded above and then closed)
# The variable name 'z' is used above as the result of np.load(NPZ_PATH).
# Since the file was closed with z.close(), let's reload it just for exploration.
with np.load(NPZ_PATH) as z_explore:
    print("Keys in npz file:", list(z_explore.keys()))
    for key in z_explore.files:
        arr = z_explore[key]
        print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")


Keys in npz file: ['embeddings', 'app_id']
embeddings: shape=(315, 512), dtype=float32
app_id: shape=(315,), dtype=int64


In [20]:
# Review the first record from the 'z' npz file
print("Sample app_id:", app_ids_npz[0])
print("Sample embedding:", X[0][:20])

Sample app_id: 70
Sample embedding: [ 0.00263368 -0.03560973 -0.03392399  0.01581145  0.00760896 -0.03232314
  0.04928112 -0.02335207  0.08172265  0.00235635  0.09670635  0.02310349
 -0.01188926 -0.0167134  -0.01041707  0.03401706 -0.03855049  0.00219529
  0.0034415  -0.03398372]


## 1) Embed query and top‑K games

Game rows are **L2-normalized** in `recs_002`, so **dot product = cosine similarity** after the query vector is normalized the same way.

In [21]:
def l2_normalize(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).ravel()
    nrm = np.linalg.norm(v)
    if nrm <= 1e-12:
        return v
    return (v / nrm).astype(np.float32)


def embed_query(text: str) -> np.ndarray:
    text = (text or "").strip()
    if MAX_CHARS is not None:
        text = text[: int(MAX_CHARS)]
    out = embed_fn([text])
    return l2_normalize(out)


def top_k_games(query: str, k: int = 10, exclude_app_ids: set[int] | None = None) -> pd.DataFrame:
    q = embed_query(query)
    scores = X @ q
    n = len(scores)
    k_eff = min(k, n)
    if exclude_app_ids:
        order_all = np.argsort(-scores)
        picked: list[int] = []
        for i in order_all:
            if int(idx_df["app_id"].iloc[int(i)]) in exclude_app_ids:
                continue
            picked.append(int(i))
            if len(picked) >= k_eff:
                break
        order = np.asarray(picked, dtype=np.int64)
    else:
        part = np.argpartition(-scores, k_eff - 1)[:k_eff]
        order = part[np.argsort(-scores[part])]
    rows = idx_df.iloc[order].copy()
    rows["score"] = scores[order]
    return rows.reset_index(drop=True)

## 2) Example queries (edit freely)

Treat scores as **cosine similarity** in \[-1, 1\] (typically positive for USE on similar prose).

In [22]:
DEMO_QUERIES = [
    "I love planes",
    "I love the realism",
    "Absolute seriousness at the center, absurdity everywhere else.",
    "Story-rich single-player RPG with choices and replay value",
    "Fast competitive FPS, skill-based, esports vibe",
    "Relaxing cozy farming or life sim, low stress",
    "Worst game ever",
    "Immersive open-world RPG with deep lore and exploration",
    "Thoughtful indie game with strong narrative",
    "Funny and lighthearted co-op party game",
    "Deep and complex tactical RPG",
    "Easy-to-learn, hard-to-master sports game",
    "Intense first-person shooter with high skill ceiling",
    "Beautiful and serene puzzle game",
    "Very very nuanced system builder",
    
]

for q in DEMO_QUERIES:
    print("\n===", q[:100], "..." if len(q) > 100 else "", "===")
    display(top_k_games(q, k=8))


=== I love planes  ===


,app_id,app_name,score
0,269950,X-Plane 11,0.363053
1,537800,Bomber Crew,0.286523
2,227300,Euro Truck Simulator 2,0.263530
3,270880,American Truck Simulator,0.235562
4,1291340,Townscaper,0.234172
5,47890,The Sims(TM) 3,0.231310
6,1118200,People Playground,0.221530
7,284160,BeamNG.drive,0.206548



=== I love the realism  ===


,app_id,app_name,score
0,1118200,People Playground,0.384251
1,284160,BeamNG.drive,0.373015
2,107410,Arma 3,0.371790
3,227300,Euro Truck Simulator 2,0.358999
4,270880,American Truck Simulator,0.344541
5,541210,Cold Waters,0.337705
6,47890,The Sims(TM) 3,0.336013
7,236510,Takedown: Red Sabre,0.335505



=== Absolute seriousness at the center, absurdity everywhere else.  ===


...

,app_id,app_name,score
0,581320,Insurgency: Sandstorm,0.469388
1,359550,Tom Clancy's Rainbow Six Siege,0.451461
2,240,Counter-Strike: Source,0.445216
3,1229490,ULTRAKILL,0.438814
4,753650,Due Process,0.424864
5,447820,Day of Infamy,0.424423
6,311690,Enter the Gungeon,0.423359
7,519860,DUSK,0.420880



=== Beautiful and serene puzzle game  ===


,app_id,app_name,score
0,288160,The Room,0.600125
1,736260,Baba Is You,0.578730
2,425580,The Room Two,0.554900
3,683320,GRIS,0.554547
4,1289310,Helltaker,0.553392
5,609320,FAR: Lone Sails,0.530431
6,1055540,A Short Hike,0.514098
7,620,Portal 2,0.513677



=== Very very nuanced system builder  ===


,app_id,app_name,score
0,352550,Urban Empire,0.271636
1,255710,Cities: Skylines,0.269892
2,621060,PC Building Simulator,0.264964
3,242920,Banished,0.242172
4,464920,Surviving Mars,0.233268
5,690830,Foundation,0.226893
6,281990,Stellaris,0.222994
7,1291340,Townscaper,0.219087


In [23]:
MY_CIV_6_REVIEW = [
    "Sean Bean's voice is great.",
    "Sean Bean's voice is great when you learn a technology",
    "Systematic city builder that stands the test of time",
    "I love working my way through the technology tree",
    "I love the combat system",
    "They added district building",
    "I love the wonders",
    "I get immense satisfaction from building natural wonders",
    "The late-game is a little bit of a grind",
]

for q in MY_CIV_6_REVIEW:
    print("\n===", q[:60], "..." if len(q) > 60 else "", "===")
    display(top_k_games(q, k=8))


=== Sean Bean's voice is great.  ===


,app_id,app_name,score
0,238460,BattleBlock Theater,0.331523
1,213670,South Park™: The Stick of Truth™,0.314099
2,420,Half-Life 2: Episode Two,0.294260
3,274190,Broforce,0.293470
4,291860,Pit People,0.293027
5,508440,Totally Accurate Battle Simulator,0.291606
6,362890,Black Mesa,0.289669
7,289070,Sid Meier's Civilization VI,0.285675



=== Sean Bean's voice is great when you learn a technology  ===


,app_id,app_name,score
0,289070,Sid Meier's Civilization VI,0.331056
1,221380,Age of Empires II (2013),0.289517
2,8930,Sid Meier's Civilization V,0.284121
3,508440,Totally Accurate Battle Simulator,0.280072
4,1017900,Age of Empires: Definitive Edition,0.278297
5,537800,Bomber Crew,0.277230
6,541210,Cold Waters,0.276592
7,284160,BeamNG.drive,0.270896



=== Systematic city builder that stands the test of time  ===


...

,app_id,app_name,score
0,1118200,People Playground,0.405752
1,688130,Pogostuck: Rage With Your Friends,0.393998
2,1291340,Townscaper,0.378854
3,238460,BattleBlock Theater,0.376415
4,240720,Getting Over It with Bennett Foddy,0.363253
5,47890,The Sims(TM) 3,0.354473
6,1055540,A Short Hike,0.345439
7,1089980,The Henry Stickmin Collection,0.340617



=== I get immense satisfaction from building natural wonders  ===


,app_id,app_name,score
0,1291340,Townscaper,0.399350
1,526870,Satisfactory,0.334932
2,255710,Cities: Skylines,0.332934
3,242920,Banished,0.321664
4,688130,Pogostuck: Rage With Your Friends,0.320103
5,323190,Frostpunk,0.316996
6,427520,Factorio,0.316961
7,690830,Foundation,0.311061



=== The late-game is a little bit of a grind  ===


,app_id,app_name,score
0,758190,Dragon Cliff 龙崖,0.454033
1,688130,Pogostuck: Rage With Your Friends,0.416358
2,677120,Heroes of Hammerwatch,0.410818
3,212680,FTL: Faster Than Light,0.410521
4,646570,Slay the Spire,0.403883
5,583950,Artifact,0.400182
6,8930,Sid Meier's Civilization V,0.398375
7,753420,Dungreed,0.395535
